# Object-oriented Python, for this thesis's codebase

Section 1 is a fast, no-fluff refresher on classes/objects/`self`/`__init__` — you've written these before, this just re-establishes them precisely rather than re-teaching from zero. Everything from Section 2 on is new ground: the dunder/"magic" methods that make your objects behave like built-in types, `@property`/`@classmethod`/`@staticmethod`, inheritance and `super()`, composition, abstract base classes, `@dataclass`, and making objects callable. These are exactly the tools your actual thesis codebase will lean on — not toy examples, but the real shapes things like `project.md`'s interpolator baseline comparison (Section 8.2) and PyG's own `Dataset`/`nn.Module` classes are built from.

**You've already used one of these patterns without a name for it.** In `pyg_tutorial.ipynb`, `TinyGNN(torch.nn.Module)` was your first taste of inheritance: subclassing a base class, calling `super().__init__()`, overriding `forward()`. This notebook makes that pattern explicit, and builds up several more like it — using the same real EGMS sample data (`data/egms_l2b_ascending_sample.csv`) so the exercises stay grounded in something you'll actually reuse.

**How this notebook works** (same convention as `pyg_tutorial.ipynb`): most code cells are `# TODO` stubs with hints and a description of what "working" looks like, not finished solutions.

```python
# TODO ---------------------------------------------------------
# <what to build>
# Hints: <pointers to docs / concepts>
# Expected: <what you should see when it works>
# ----------------------------------------------------------------
```

Fill those in yourself. A couple of cells (data loading) are given outright because they're plumbing, not the skill you're here to build.

**Reference docs you'll come back to:**
- Python data model (all the dunder methods): https://docs.python.org/3/reference/datamodel.html
- `dataclasses`: https://docs.python.org/3/library/dataclasses.html
- `abc` (abstract base classes): https://docs.python.org/3/library/abc.html
- Real Python, "Inheritance and Composition": https://realpython.com/inheritance-composition-python/
- Real Python, "Python's `super()`": https://realpython.com/python-super/


## 0. Setup

**Given:** load the same EGMS sample used in `pyg_tutorial.ipynb` — 4772 real points near Munich, one `mean_velocity` (mm/yr, LOS) per point. Everything below builds classes around this one flat DataFrame.

In [ ]:
import abc
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from pyproj import Geod

df = pd.read_csv("data/egms_l2b_ascending_sample.csv")
GEOD = Geod(ellps="WGS84")  # same geodesic-distance tool as pyg_tutorial Section 5

print(df.shape)
df[["pid", "latitude", "longitude", "mean_velocity"]].head()


## 1. OOP basics, refreshed

Fast and precise, not a first introduction — if any line below is new information rather than a reminder, slow down here before going further.

- **Class vs. instance.** A class is a blueprint; instantiating it (`DisplacementPoint(...)`) builds one concrete object from that blueprint. Nothing runs "just from defining the class" — the body only executes once, at class-creation time, to set up the class itself.
- **`self`.** `obj.method(x)` is syntactic sugar for `ClassName.method(obj, x)` — `self` *is* the specific instance, passed automatically as the first parameter. It's a parameter name by convention, not a keyword; call it something else and it still works identically (don't, though — every Python codebase you'll ever read assumes `self`).
- **`__init__`.** Runs once, immediately after a new instance is created, and its job is entirely side effects: set up `self`'s starting state. It returns `None` implicitly — `__init__` never `return`s a value.
- **Instance attributes vs. class attributes — the gotcha worth knowing precisely.** `self.x = 5` inside `__init__` creates an attribute that belongs to *that one instance*. A name assigned directly in the class body (not inside any method) belongs to the *class*, and is **shared by every instance** until an instance sets its own attribute of the same name. This is harmless for immutable values (an `int`, a `str`) but a classic bug for mutable defaults (a `list` or `dict` defined at class level and then `.append()`-ed to from an instance method — every instance mutates the *same* shared list). We'll use a class attribute for something genuinely shared (a constant, not mutable state) below.
- **Leading underscore = "internal".** `self._df` (you'll write this in Exercise 2) signals "implementation detail, not part of this class's public interface" — Python doesn't enforce it, unlike a truly private attribute in some other languages; it's a convention every reader is expected to respect.

### Exercise 1
`DisplacementPoint`:
- a **class attribute** `VELOCITY_UNIT = "mm/yr"` — one value, shared by every instance, because it's a constant fact about the data, not something that varies per point
- `__init__(self, pid, lat, lon, mean_velocity)` setting the four as **instance** attributes
- a method `is_stable(self, threshold=1.0)` returning whether `abs(self.mean_velocity) < threshold`

Then: construct one instance from row 0 of `df`, and print `mean_velocity` alongside `VELOCITY_UNIT` accessed two ways — `DisplacementPoint.VELOCITY_UNIT` (via the class) and `p0.VELOCITY_UNIT` (via the instance) — to see for yourself that both reach the same shared value.

In [ ]:
# TODO ---------------------------------------------------------
# Expected: p0.pid == df.loc[0, "pid"]; p0.is_stable() is a bool;
#           DisplacementPoint.VELOCITY_UNIT == p0.VELOCITY_UNIT == "mm/yr".
# ----------------------------------------------------------------

class DisplacementPoint:
    <...>  # class attribute here

    def __init__(self, pid, lat, lon, mean_velocity):
        <...>

    def is_stable(self, threshold=1.0):
        <...>

p0 = <...>
print(p0.pid, p0.mean_velocity, p0.is_stable())
print(DisplacementPoint.VELOCITY_UNIT, p0.VELOCITY_UNIT)


## 2. Dunder methods — making your object act like a built-in

Python doesn't have one universal "sequence" interface bolted onto every class — it's convention: implement `__len__` and `len(obj)` works, implement `__getitem__` and `obj[i]` and `for x in obj` both work (Python falls back to integer-indexed iteration via `__getitem__` if you don't give it a separate `__iter__`). This is *exactly* the protocol `torch.utils.data.Dataset` relies on — a PyG/PyTorch `Dataset` is just a class with `__len__` and `__getitem__`, nothing more magic than what you're about to build by hand.

### Exercise 2
`EGMSPointSet` wrapping the whole DataFrame:
- `__init__(self, dataframe)` — store it (reset the index so positions 0..N-1 are contiguous)
- `__len__` — number of rows
- `__getitem__(self, idx)` — return a `DisplacementPoint` for row `idx` (reuse Exercise 1's class)
- `__repr__` — something readable, e.g. `EGMSPointSet(4772 points)`, so it prints usefully instead of `<__main__.EGMSPointSet object at 0x...>`
- `__iter__` — a generator yielding each point in order (don't rely only on the `__getitem__` fallback — write this one explicitly so you've seen the pattern)

**Hint:** `dataframe.iloc[idx]` gets you a row by position; pull the four fields out of it to build the `DisplacementPoint`.

In [ ]:
# TODO ---------------------------------------------------------
# Expected: len(pset) == 4772; pset[0].pid == df.loc[0, "pid"];
#           repr(pset) prints something readable;
#           sum(1 for _ in pset) == len(pset)
# ----------------------------------------------------------------

class EGMSPointSet:
    def __init__(self, dataframe):
        <...>

    def __len__(self):
        <...>

    def __getitem__(self, idx):
        <...>

    def __repr__(self):
        <...>

    def __iter__(self):
        <...>

pset = EGMSPointSet(df)
print(repr(pset), len(pset), pset[0].pid)
print(sum(1 for _ in pset) == len(pset))


## 3. `@property`, `@classmethod`, `@staticmethod`

Three different "not quite a normal method" decorators, three different jobs:
- **`@property`** — a method you call *without* parentheses (`point.velocity_category`, not `point.velocity_category()`). Use it for a value that's cheap to derive from existing attributes and conceptually *is* an attribute, just a computed one.
- **`@classmethod`** — receives the class (`cls`) instead of an instance (`self`). The standard use is an alternate constructor: `EGMSPointSet.from_csv(path)` instead of `EGMSPointSet(pd.read_csv(path))`.
- **`@staticmethod`** — no `self`, no `cls`. It's a plain function that just happens to live inside the class because it's conceptually related. Use it when the method doesn't touch instance or class state at all.

### Exercise 3
1. Add a `velocity_category` **property** to `DisplacementPoint`: `"subsiding"` if `mean_velocity < -1.0`, `"uplifting"` if `> 1.0`, else `"stable"`.
2. Add a **staticmethod** `geodesic_distance_m(lat1, lon1, lat2, lon2)` to `DisplacementPoint` — reuse `GEOD.inv(lon1, lat1, lon2, lat2)` (same call shape as `pyg_tutorial.ipynb` Section 5) and return just the distance.
3. Add a **classmethod** `from_csv(cls, path)` to `EGMSPointSet` that builds an instance directly from a CSV path.

You'll extend the classes from Exercises 1–2 in place — that's normal, classes grow as you need more from them.

In [ ]:
# TODO ---------------------------------------------------------
# Expected: pset[0].velocity_category in {"subsiding","stable","uplifting"};
#           DisplacementPoint.geodesic_distance_m(48.0, 11.0, 48.01, 11.0) is a
#           plausible distance in metres (order of ~1000, it is ~0.01 deg of latitude);
#           EGMSPointSet.from_csv("data/egms_l2b_ascending_sample.csv") works
#           without reading the CSV yourself first.
# ----------------------------------------------------------------

class DisplacementPoint:
    VELOCITY_UNIT = "mm/yr"

    def __init__(self, pid, lat, lon, mean_velocity):
        self.pid = pid
        self.lat = lat
        self.lon = lon
        self.mean_velocity = mean_velocity

    def is_stable(self, threshold=1.0):
        return abs(self.mean_velocity) < threshold

    @property
    def velocity_category(self):
        <...>

    @staticmethod
    def geodesic_distance_m(lat1, lon1, lat2, lon2):
        <...>


class EGMSPointSet:
    def __init__(self, dataframe):
        self._df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self._df)

    def __getitem__(self, idx):
        row = self._df.iloc[idx]
        return DisplacementPoint(row["pid"], row["latitude"], row["longitude"], row["mean_velocity"])

    def __repr__(self):
        return f"EGMSPointSet({len(self)} points)"

    def __iter__(self):
        for i in range(len(self)):
            yield self[i]

    @classmethod
    def from_csv(cls, path):
        <...>


pset = EGMSPointSet.from_csv("data/egms_l2b_ascending_sample.csv")
print(pset[0].velocity_category)
print(DisplacementPoint.geodesic_distance_m(48.0, 11.0, 48.01, 11.0))


## 4. Inheritance & `super()` — a common interface, different behaviour

`project.md` Section 8.2 needs to compare your GNN against a regression-kriging baseline **fairly** — same neighbours, same query points, only the interpolation method itself differs. That's inheritance's actual job: define the shared shape once in a base class, let subclasses swap out only the part that differs.

### Exercise 4
- `Interpolator`: `__init__(self, points: EGMSPointSet)` stores `self.points = points`. A method `predict(self, lat, lon, k=1)` that just does `raise NotImplementedError` — the base class defines the *interface*, not a working method.
- `NearestNeighborInterpolator(Interpolator)`: inherits `__init__` as-is (don't rewrite it — that's the point of inheritance), overrides `predict` to find the single closest point (by `DisplacementPoint.geodesic_distance_m`, looping over `self.points`) and return its `mean_velocity`.

**Hint:** looping in Python over all 4772 points to find a minimum is fine here (clarity over speed — Exercise 5 fixes the speed).

In [ ]:
# TODO ---------------------------------------------------------
# Expected: NearestNeighborInterpolator(pset).predict(48.1, 11.6) returns a
#           float close to some real mean_velocity in the sample.
# ----------------------------------------------------------------

class Interpolator:
    def __init__(self, points):
        <...>

    def predict(self, lat, lon, k=1):
        <...>


class NearestNeighborInterpolator(Interpolator):
    def predict(self, lat, lon, k=1):
        <...>


nn_interp = NearestNeighborInterpolator(pset)
print(nn_interp.predict(48.1, 11.6))


## 5. Composition — "has a" instead of "is a"

Every `Interpolator` subclass is going to need "find the k nearest points to (lat, lon)" — that's not really part of *being* an interpolation method, it's a separate, reusable piece of geometry. The rule of thumb: if a class would need to duplicate logic that isn't really about what that class *is*, pull that logic into its own class and **hold an instance of it as an attribute** instead of copy-pasting or re-inheriting it. This is "favour composition over inheritance" — inheritance answers "what kind of thing is this", composition answers "what does this thing use".

### Exercise 5
`NeighborFinder`:
- `__init__(self, points: EGMSPointSet)` — store `points`, and pull `latitude`/`longitude` out once as numpy arrays (don't re-extract them on every query — that's the speed fix from Exercise 4's hint)
- `query(self, lat, lon, k=10, exclude_idx=None)` — vectorised geodesic distance from `(lat, lon)` to every stored point in one `GEOD.inv(...)` call (same shape as `pyg_tutorial.ipynb` Section 5's `build_query_graph`), `np.argsort` to rank them, optionally drop `exclude_idx` from the ranking (you'll need this in Exercise 8 — think about why), then return the `k` nearest as `(indices, distances)`.

Then refactor `NearestNeighborInterpolator` to **hold** a `NeighborFinder` (built once in `__init__`) instead of looping by hand.

In [ ]:
# TODO ---------------------------------------------------------
# Expected: nf.query(48.1, 11.6, k=5) returns 5 indices and 5 distances,
#           sorted ascending; refactored NearestNeighborInterpolator gives
#           the same prediction as Exercise 4 but should run noticeably
#           faster over many calls.
# ----------------------------------------------------------------

class NeighborFinder:
    def __init__(self, points):
        self.points = points
        self._lats = <...>
        self._lons = <...>

    def query(self, lat, lon, k=10, exclude_idx=None):
        <...>


class Interpolator:
    def __init__(self, points):
        self.points = points
        self.finder = NeighborFinder(points)   # composition: Interpolator HAS a NeighborFinder

    def predict(self, lat, lon, k=1):
        raise NotImplementedError


class NearestNeighborInterpolator(Interpolator):
    def predict(self, lat, lon, k=1, exclude_idx=None):
        <...>


nf = NeighborFinder(pset)
idx, dist = nf.query(48.1, 11.6, k=5)
print(idx, dist)

nn_interp = NearestNeighborInterpolator(pset)
print(nn_interp.predict(48.1, 11.6))


## 6. Abstract base classes — making the interface unskippable

`Interpolator.predict` raising `NotImplementedError` is a *convention* — nothing stops someone from instantiating bare `Interpolator()` and only finding out it's broken when `predict` is actually called, possibly deep inside a training loop. `abc.ABC` + `@abc.abstractmethod` turns that into an enforced rule: Python refuses to construct a class that hasn't overridden every abstract method, at the moment you try to instantiate it.

This is the real shape of `project.md` Section 8.2's baseline comparison: one `Interpolator` interface, several interchangeable implementations (nearest-neighbour, IDW, regression-kriging, your GNN), all swappable in the exact same evaluation loop.

### Exercise 6
1. Make `Interpolator` inherit from `abc.ABC` and mark `predict` with `@abc.abstractmethod`.
2. Add `IDWInterpolator(Interpolator)`: `__init__(self, points, power=2.0)` (call `super().__init__(points)` first, then store `power`), `predict` does inverse-distance weighting over the `k` nearest neighbours from `self.finder` — same formula as `presentation/fig09_summary_bg_idw.py` if you want to see it applied to a whole grid instead of one point: `weight = distance ** -power`, prediction = weighted average of `mean_velocity`. Guard against a zero distance (`np.maximum(dist, 1.0)`) so a point landing exactly on a sample doesn't divide by zero.

**Verify the enforcement actually works:** try instantiating bare `Interpolator(pset)` and confirm it raises `TypeError`.

In [ ]:
# TODO ---------------------------------------------------------
# Expected: Interpolator(pset) raises TypeError (abstract method not implemented);
#           IDWInterpolator(pset).predict(48.1, 11.6, k=10) returns a float.
# ----------------------------------------------------------------

class Interpolator(<...>):
    def __init__(self, points):
        self.points = points
        self.finder = NeighborFinder(points)

    @<...>
    def predict(self, lat, lon, k=1, exclude_idx=None):
        ...


class IDWInterpolator(Interpolator):
    def __init__(self, points, power=2.0):
        <...>

    def predict(self, lat, lon, k=10, exclude_idx=None):
        <...>


try:
    Interpolator(pset)
    print("did NOT raise - check your @abstractmethod")
except TypeError as e:
    print("correctly blocked:", e)

idw_interp = IDWInterpolator(pset, power=2.0)
print(idw_interp.predict(48.1, 11.6, k=10))


## 7. `@dataclass` — plain data, without the boilerplate

Compare writing `__init__` by hand (assign every field to `self`) against `@dataclass`, which generates `__init__`, `__repr__`, and `__eq__` for you from type-annotated class attributes. Use it for objects that are mostly *data* — configuration, records — as opposed to `Interpolator`, which is mostly *behaviour*.

### Exercise 7
`QueryConfig`: `k: int = 10`, `power: float = 2.0`, `mask_rate: float = 0.25` (the same neighbour-dropout knob as `pyg_tutorial.ipynb` Section 7). Add `__post_init__` (dataclasses call this automatically right after the generated `__init__`) that asserts `k > 0` and `0.0 <= mask_rate < 1.0`, raising a clear error otherwise.

**Hint:** the decorator is `@dataclass` above the class; fields are just `name: type = default` lines, no `self`, no `__init__` to write.

In [ ]:
# TODO ---------------------------------------------------------
# Expected: QueryConfig() prints a readable repr with all three defaults;
#           QueryConfig(k=-1) raises AssertionError from __post_init__.
# ----------------------------------------------------------------

@<...>
class QueryConfig:
    k: <...>
    power: <...>
    mask_rate: <...>

    def __post_init__(self):
        <...>


print(QueryConfig())
try:
    QueryConfig(k=-1)
    print("did NOT raise - check __post_init__")
except AssertionError as e:
    print("correctly rejected:", e)


## 8. `__call__` — an object that acts like a function

Implement `__call__` and an instance can be used with `()` just like a function — but unlike a plain function, it can carry state set up once and reused across many calls. That's useful exactly when "the function" needs configuration that would otherwise mean passing the same arguments everywhere.

### Exercise 8
`RMSEScorer`: `__init__(self, query_points: EGMSPointSet, k=10, seed=0, n_eval=200)` picks `n_eval` random indices from `query_points` **once** (`np.random.default_rng(seed).choice(...)`) and stores them — that's the state. `__call__(self, interpolator)` loops over the stored indices, and for each one, predicts at that point's `(lat, lon)` and compares to its real `mean_velocity`, returning the RMSE over all of them.

**The gotcha to get right:** if you evaluate a point that's still sitting inside `interpolator.points`, its own nearest neighbour is *itself*, at distance 0 — a free, meaningless perfect prediction. Pass `exclude_idx=i` through to `interpolator.predict(...)` (you built this into `NeighborFinder.query` back in Exercise 5) so each evaluated point can't see itself. This is the same principle as `pyg_tutorial.ipynb`'s masked-target/query-anywhere design (`project.md` Section 3.2) — a model (or here, a hand-built interpolator) only looks honest if it never gets to see the answer it's being scored on.

In [ ]:
# TODO ---------------------------------------------------------
# Expected: RMSEScorer(pset, k=10, seed=0)(nn_interp) and (idw_interp) both
#           return floats; the NN one should be noticeably worse (higher)
#           than IDW at k=10 - averaging several neighbours beats picking one.
# ----------------------------------------------------------------

class RMSEScorer:
    def __init__(self, query_points, k=10, seed=0, n_eval=200):
        <...>

    def __call__(self, interpolator):
        <...>


score = RMSEScorer(pset, k=10, seed=0)
print("NN  RMSE:", score(nn_interp))
print("IDW RMSE:", score(idw_interp))


## 9. Capstone — the fair bake-off

This is `project.md` Section 8.2 in miniature: one shared `Interpolator` interface, several concrete implementations, one `RMSEScorer` evaluating all of them identically. Nothing new to build — this cell just exercises everything above together.

### Exercise 9
Compare `NearestNeighborInterpolator` against `IDWInterpolator` at a few different `k` values (try `k=3, 5, 10, 20`, passed via `QueryConfig`), using the same `RMSEScorer` instance for every comparison so the evaluation points are identical across runs. Print a small table of `k -> RMSE`.

Once this works, the honest next step (not required here, just worth seeing): a `GNNInterpolator(Interpolator)` wrapping `pyg_tutorial.ipynb`'s `TinyGNN` — `predict()` would build a query graph the same way `build_query_graph_indkrig` does, run the model, and return `model(graph).item()`. Same interface, same `RMSEScorer`, directly comparable numbers — that fairness is the entire point of building `Interpolator` as an ABC instead of three unrelated functions.

In [ ]:
# TODO ---------------------------------------------------------
# Expected: a printed table showing IDW RMSE roughly flattening out or
#           slightly improving as k grows, and beating NearestNeighbor
#           (k=1 has no averaging to smooth out noise).
# ----------------------------------------------------------------

scorer = RMSEScorer(pset, k=10, seed=0)

print("NearestNeighbor:", scorer(NearestNeighborInterpolator(pset)))

for k in [3, 5, 10, 20]:
    cfg = QueryConfig(k=k)
    scorer_k = RMSEScorer(pset, k=cfg.k, seed=0)
    idw_k = IDWInterpolator(pset, power=cfg.power)
    print(f"IDW k={cfg.k}:", <...>)


## 10. Where this leaves you

| Notebook section | Pattern | Where it shows up for real |
|---|---|---|
| 2 | `__len__`/`__getitem__`/`__iter__` | this is what `torch.utils.data.Dataset` / PyG `Dataset` actually are |
| 3 | `@property`/`@classmethod`/`@staticmethod` | alternate constructors, derived read-only fields |
| 4 | inheritance, `super()` | `TinyGNN(torch.nn.Module)` in `pyg_tutorial.ipynb`, any `Interpolator` subclass |
| 5 | composition | `Interpolator` holding a `NeighborFinder`; a real model holding sub-modules |
| 6 | `abc.ABC` | enforcing one interface across your GNN + the regression-kriging baseline (`project.md` 8.2) |
| 7 | `@dataclass` | config objects — `QueryConfig` here, `HANDPICKED_REGIONS`-style records elsewhere in this repo |
| 8 | `__call__` | a scorer/loss with state, instead of a bare function needing the same arguments passed everywhere |

**Deliberately not covered** — each is a real thing, just not needed yet:
- **Multiple inheritance / mixins.** Rare in practice once composition is available; the `Interpolator` hierarchy here never needed it.
- **Operator overloading** (`__add__`, `__eq__`, etc. beyond what `@dataclass` generates for you). Useful for things you want to add/compare directly (e.g. combining two displacement vectors) — worth knowing exists, not used here.
- **Descriptors and metaclasses.** What `@property` and `@dataclass` are actually built out of, one level down. You don't need to write your own to use them.

If you want to see this same shape used at production scale rather than on a 4772-point toy sample, `torch_geometric.data.Dataset` and `torch_geometric.nn.MessagePassing` (both classes you've already been *using* in `pyg_tutorial.ipynb`) are exactly `__getitem__`/`__len__` and ABC-with-a-required-`forward`, respectively — go look at their source now and it should read as familiar rather than magic.
